# Feature Engineering

## Import modules

In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
import scipy.sparse as sp
import joblib

## Import data

In [2]:
df = pd.read_parquet('csv/preprocess.parquet')
df.head()

,Sentiment,Text Content,tweet_length,Mutated Text Content,Text Tokens
0,1,Love Going to a Depot or a Low Coast,36,love going to a depot or a low coast,"[love, going, depot, low, coast]"
1,0,Borderlands 3 Update Delayed Due To Anti-Racis...,83,borderlands 3 update delayed due to anti-racis...,"[borderlands, 3, update, delayed, due, antirac..."
2,1,almost done YES YES tomorrow we can finish our...,107,almost done yes yes tomorrow we can finish our...,"[almost, done, yes, yes, tomorrow, finish, goa..."
3,1,The new cod made..actually really fun.,38,the new cod made..actually really fun.,"[new, cod, made, actually, really, fun]"
4,0,@JeffBezos Fuck Amazon. Love your body better.,46,@jeffbezos fuck amazon. love your body better.,"[fuck, amazon, love, body, better]"


# TF-IDF

TF-IDF (term frequency - inverse document frequency) is used in natural language processing models (like the one I am making) to calculate the importance of a word in a document/block of text by using statistics. It is used to extract the features (like the value of a word compared to its sentiment). 

## TF
TF (term frequency) measures how often a word shows up in a document. A higher frequency suggests greater importance. If it shows up frequently, it likely means it is highly relevant to the document. 
<!-- For anyone viewing the source of this, I used [GFG](https://www.geeksforgeeks.org/machine-learning/understanding-tf-idf-term-frequency-inverse-document-frequency/) for the definitions -->

## IDF
IDF is the inverse document frequency and reduces the weight of common words within the document while increasing the weight of rare words. If a term (such as "this") appears more likely than ("newtonian"), it is less important, while "newtonian" is more important and its weighting gets raised.  

## Calculating
Calculating is simple, as it is simply the product of the two:
$$
\text{TF-IDF(t, d, D)} = \text{TF(t, d)} \times \text{IDF(t, D)}
$$

# Showing as code

We use a type of TF-IDF feature extractor called TfidfVectoriser, which just takes a word from our tokenised values (also done in preprocessing) and passes it into the TF-IDF function, which returns a weighting for the word. 

This is then stored as a feature matrix (rows = tweets/documents, columns = vocabulary, values = TF-IDF weights). This can then be used as input for Logistic Regression (which is done in model training). 

While training my vectoriser, I have come to an issue that 8 million rows will take up too much memory. 

I have decided to instead take a sample of `500,000` rows instead of using the entire thing, that way I can use 16x the amount of memory for a similar accuracy. 

In [3]:
df = df.sample(n=1_000_000, random_state=42).reset_index(drop=True)

In [4]:
tfidf = TfidfVectorizer(
    ngram_range=(1, 2) # allows for such features like "not good"
)
X_tfidf = tfidf.fit_transform(df['Mutated Text Content'])

X = sp.hstack([X_tfidf])
y = df['Sentiment']

Within the vectoriser, I have used a ngram_range as an argument. This combines tokens if the context is relevant, and allows for features like ["not good"] instead of ["not", "good"], which might lose context with the negator. 

Within this data, we have got 5000 features (as specified by the TfidfVectorizer) and have fitted the imported CSV `Text Tokens` column as the fit. 

The Y is the `Sentiment` column. 

# Exporting

I think it is ready to export the vectoriser for training to use. 

In [5]:
joblib.dump(tfidf, 'csv/tfidf_dump.pkl')
sp.save_npz('csv/x_features.npz', X)
y.to_csv('csv/y_sentiment_labels.csv', index=False)